# Model Evaluation & Clinical Thresholding

## Objective
Evaluate the trained Logistic Regression model with a clinical focus on recall (sensitivity) and determine a safe probability threshold for cardiac risk prediction.

## Why This Matters
In cardiac risk screening, false negatives (missed high-risk patients) are clinically more dangerous than false positives. This notebook analyzes model performance accordingly.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    f1_score,
    recall_score,
    precision_score
)
import joblib

# Configure visualization
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [5]:
# File paths (centralized for clarity)
import os

# Get the notebook directory and navigate to project root
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

X_PATH = os.path.join(DATA_DIR, "X_scaled.csv")
Y_PATH = os.path.join(DATA_DIR, "y.csv")
MODEL_PATH = os.path.join(DATA_DIR, "logistic_regression_model.pkl")

print(f"Data directory: {DATA_DIR}")
print(f"X path: {X_PATH}")
# This avoids hardcoding paths everywhere (production-grade habit).


Data directory: c:\Users\Pc\Desktop\cardiac-risk-awareness\data\processed
X path: c:\Users\Pc\Desktop\cardiac-risk-awareness\data\processed\X_scaled.csv


In [6]:
# Load data
X = pd.read_csv(X_PATH)
y = pd.read_csv(Y_PATH).squeeze()  # ensure y is a Series

# Load trained model
model = joblib.load(MODEL_PATH)

print("Shapes:")
print("X:", X.shape)
print("y:", y.shape)


Shapes:
X: (5160, 44)
y: (5160,)


In [7]:
# Recreate train-validation split (same logic as training)
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)


Train shape: (4128, 44) (4128,)
Validation shape: (1032, 44) (1032,)


In [8]:
# Predict class labels
# Convert all columns to numeric, coercing errors to NaN then filling
X_val_numeric = X_val.copy()
for col in X_val_numeric.columns:
    X_val_numeric[col] = pd.to_numeric(X_val_numeric[col], errors='coerce')

# Fill any NaN values with 0
X_val_numeric = X_val_numeric.fillna(0)

y_val_pred = model.predict(X_val_numeric)

# Predict probabilities for positive (cardiac risk) class
y_val_proba = model.predict_proba(X_val_numeric)[:, 1]

print("Predictions shape:", y_val_pred.shape)
print("Probabilities shape:", y_val_proba.shape)
print("Unique predictions:", np.unique(y_val_pred))


Predictions shape: (1032,)
Probabilities shape: (1032,)
Unique predictions: [0. 1.]


In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score

metrics = {
    "Recall (Sensitivity)": recall_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred),
    "F1 Score": f1_score(y_val, y_val_pred),
    "ROC-AUC": roc_auc_score(y_val, y_val_proba)
}

metrics_df = pd.DataFrame.from_dict(metrics, orient="index", columns=["Value"])
metrics_df


,Value
Recall (Sensitivity),0.636364
Precision,0.434911
F1 Score,0.516696
ROC-AUC,0.782912


## Clinical Context: Why Recall is Prioritized Over Accuracy

### Why Recall is Prioritized in Cardiac Screening
In cardiac risk screening, the primary goal is to identify as many at-risk patients as possible.

Accuracy can be misleading when classes are imbalanced, but **high recall ensures fewer high-risk individuals are missed**, which is critical in preventive healthcare.

### What a False Negative Means Clinically
A **false negative** means the model predicts a patient is low risk when they are actually at high cardiac risk.

Clinically, this could:
- Delay medical intervention
- Prevent necessary lifestyle changes
- Postpone further diagnostic testing
- Potentially lead to serious or fatal outcomes

### Why ROC-AUC is Still Reported
**ROC-AUC** measures the model's overall ability to discriminate between high-risk and low-risk patients across all thresholds.

Reporting ROC-AUC helps:
- Assess whether the model is fundamentally sound
- Understand performance across different decision thresholds
- Ensure the model isn't just lucky with the current threshold choice
- Support evidence-based threshold selection for deployment

## Confusion Matrix

In [10]:
from sklearn.metrics import confusion_matrix
import seaborn as sns 
import matplotlib.pyplot as plt

In [ ]:
# Preprocess X_val: Encode categorical columns & ensure all numeric
from sklearn.preprocessing import LabelEncoder

X_val_prep = X_val.copy()

# Identify and encode categorical columns
categorical_cols = X_val_prep.select_dtypes(include=['object']).columns.tolist()

print("Categorical columns detected:", categorical_cols)
print("\nDatatype before encoding:")
print(X_val_prep.dtypes)

# Label encode each categorical column
label_encoders_eval = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_val_prep[col] = le.fit_transform(X_val_prep[col].astype(str))
    label_encoders_eval[col] = le
    print(f"✓ Encoded '{col}' — classes: {list(le.classes_)}")

print("\nDatatype after encoding:")
print(X_val_prep.dtypes)
print("\nX_val_prep shape:", X_val_prep.shape)
print("All numeric? ", X_val_prep.dtypes.apply(lambda x: x in ['int64', 'float64']).all())

## Generate predictions on preprocessed validation data

In [ ]:
# Use the preprocessed, encoded validation data for predictions
y_pred = model.predict(X_val_prep)
y_pred_proba = model.predict_proba(X_val_prep)[:, 1]

print("Predictions generated successfully!")
print(f"Prediction shape: {y_pred.shape}")
print(f"Prediction distribution: {pd.Series(y_pred).value_counts().to_dict()}")
print(f"Probability range: [{y_pred_proba.min():.4f}, {y_pred_proba.max():.4f}]")

TypeError: can't multiply sequence by non-int of type 'float'

In [12]:
print(X_val.dtypes)

BMI                        float64
BPMeds                       int64
age                        float64
ca                         float64
cigsPerDay                 float64
cp                             str
currentSmoker                int64
diaBP                      float64
diabetes                     int64
education                  float64
exang                        int64
fbs                          int64
glucose                    float64
heartRate                  float64
oldpeak                    float64
prevalentHyp                 int64
prevalentStroke              int64
restecg                        str
sex                            str
slope                          str
sysBP                      float64
thal                           str
totChol                    float64
BMI_missing                  int64
BPMeds_missing               int64
ca_missing                   int64
cigsPerDay_missing           int64
cp_missing                   int64
currentSmoker_missin

In [13]:
print(X_val.head())

           BMI  BPMeds       age  ...  sysBP_missing  thal_missing totChol_missing
4528  0.777767       0  1.435909  ...              0             1               0
4204 -1.142186       0 -1.273568  ...              0             1               0
380  -0.088785       0 -0.596199  ...              0             1               0
835  -0.088785       0  1.548804  ...              1             1               0
1252  0.896917       0  1.210119  ...              0             1               0

[5 rows x 44 columns]
